In [1]:
import os
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import joblib

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
PROJECT_ROOT = r"D:\TrustXAI"

DATA_FILE = os.path.join(
    PROJECT_ROOT,
    "data",
    "ready",
    "TrustXAI_Encoded.csv"
)

MODELS_PATH = os.path.join(PROJECT_ROOT, "models")

os.makedirs(MODELS_PATH, exist_ok=True)

print(DATA_FILE)

D:\TrustXAI\data\ready\TrustXAI_Encoded.csv


In [3]:
print("Reading dataset labels...")

CHUNK_SIZE = 100000

label_counts = {}

for chunk in pd.read_csv(
    DATA_FILE,
    usecols=["Label"],
    chunksize=CHUNK_SIZE
):

    counts = chunk["Label"].value_counts()

    for label, count in counts.items():

        label_counts[label] = label_counts.get(label, 0) + count

print("\nClass Distribution:\n")

for label in sorted(label_counts.keys()):
    print(f"Label {label}: {label_counts[label]:,}")

Reading dataset labels...

Class Distribution:

Label 0: 2,271,320
Label 1: 1,956
Label 2: 128,025
Label 3: 10,293
Label 4: 230,124
Label 5: 5,499
Label 6: 5,796
Label 7: 7,935
Label 8: 11
Label 9: 36
Label 10: 158,804
Label 11: 5,897
Label 12: 1,507
Label 13: 21
Label 14: 652


In [4]:
import os
import pandas as pd

# ===============================
# Create Balanced Dataset
# ===============================

OUTPUT_FILE = os.path.join(
    PROJECT_ROOT,
    "data",
    "ready",
    "TrustXAI_Balanced.csv"
)

CHUNK_SIZE = 100000
MAX_SAMPLES_PER_CLASS = 5000

rare_classes = [8, 9, 13]

saved_rows = {}


def remaining_space(label):
    return MAX_SAMPLES_PER_CLASS - saved_rows.get(label, 0)


first_chunk = True

print("Creating balanced dataset...\n")

for chunk in pd.read_csv(DATA_FILE, chunksize=CHUNK_SIZE, low_memory=False):

    # Remove rare classes
    chunk = chunk[~chunk["Label"].isin(rare_classes)]

    selected_parts = []

    for label in sorted(chunk["Label"].unique()):

        remain = remaining_space(label)

        if remain <= 0:
            continue

        subset = chunk[chunk["Label"] == label]

        if len(subset) > remain:
            subset = subset.sample(
                n=remain,
                random_state=42
            )

        saved_rows[label] = saved_rows.get(label, 0) + len(subset)

        selected_parts.append(subset)

    if len(selected_parts) == 0:
        continue

    balanced_chunk = pd.concat(selected_parts)

    balanced_chunk.to_csv(
        OUTPUT_FILE,
        mode="w" if first_chunk else "a",
        header=first_chunk,
        index=False
    )

    first_chunk = False

    print(saved_rows)

    if all(v >= MAX_SAMPLES_PER_CLASS for v in saved_rows.values()):
        if len(saved_rows) >= 12:
            break

print("\nBalanced dataset created successfully.")
print(OUTPUT_FILE)

Creating balanced dataset...

{np.int64(0): 5000, np.int64(2): 5000}
{np.int64(0): 5000, np.int64(2): 5000, np.int64(10): 372}
{np.int64(0): 5000, np.int64(2): 5000, np.int64(10): 5000}
{np.int64(0): 5000, np.int64(2): 5000, np.int64(10): 5000, np.int64(1): 202}
{np.int64(0): 5000, np.int64(2): 5000, np.int64(10): 5000, np.int64(1): 1849}
{np.int64(0): 5000, np.int64(2): 5000, np.int64(10): 5000, np.int64(1): 1956}
{np.int64(0): 5000, np.int64(2): 5000, np.int64(10): 5000, np.int64(1): 1956, np.int64(12): 1507, np.int64(14): 338}
{np.int64(0): 5000, np.int64(2): 5000, np.int64(10): 5000, np.int64(1): 1956, np.int64(12): 1507, np.int64(14): 652}
{np.int64(0): 5000, np.int64(2): 5000, np.int64(10): 5000, np.int64(1): 1956, np.int64(12): 1507, np.int64(14): 652, np.int64(7): 5000}
{np.int64(0): 5000, np.int64(2): 5000, np.int64(10): 5000, np.int64(1): 1956, np.int64(12): 1507, np.int64(14): 652, np.int64(7): 5000, np.int64(11): 2973}
{np.int64(0): 5000, np.int64(2): 5000, np.int64(10): 50

In [5]:
# =====================================
# Load Balanced Dataset
# =====================================

BALANCED_FILE = os.path.join(
    PROJECT_ROOT,
    "data",
    "ready",
    "TrustXAI_Balanced.csv"
)

df = pd.read_csv(BALANCED_FILE)

print(df.shape)

(49115, 85)


In [6]:
# =====================================
# Remove Text Columns
# =====================================

columns_to_drop = [
    "Flow ID",
    "Source IP",
    "Destination IP",
    "Timestamp"
]

X = df.drop(columns=columns_to_drop + ["Label"])

y = df["Label"]

print("Features:", X.shape)
print("Labels:", y.shape)

Features: (49115, 80)
Labels: (49115,)


In [7]:
# =====================================
# Train Test Split
# =====================================

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape)
print("Test :", X_test.shape)

Train: (39292, 80)
Test : (9823, 80)


In [8]:
# =====================================
# Improved Random Forest
# =====================================

rf_model = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

print("Training...")

rf_model.fit(X_train, y_train)

print("Training Finished.")

Training...
Training Finished.


In [9]:
# =====================================
# Prediction
# =====================================

y_pred = rf_model.predict(X_test)

print("Prediction completed.")

Prediction completed.


In [10]:
# =====================================
# Model Evaluation
# =====================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average="weighted")
recall = recall_score(y_test, y_pred, average="weighted")
f1 = f1_score(y_test, y_pred, average="weighted")

print("=" * 50)
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print("=" * 50)

Accuracy : 0.9900
Precision: 0.9901
Recall   : 0.9900
F1 Score : 0.9900


In [11]:
# =====================================
# Classification Report
# =====================================

from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      0.99      0.99      1000
           1       0.99      1.00      1.00       391
           2       1.00      1.00      1.00      1000
           3       1.00      1.00      1.00      1000
           4       1.00      1.00      1.00      1000
           5       1.00      0.99      0.99      1000
           6       0.99      0.99      0.99      1000
           7       1.00      1.00      1.00      1000
          10       1.00      1.00      1.00      1000
          11       1.00      1.00      1.00      1000
          12       0.85      0.91      0.88       301
          14       0.76      0.69      0.72       131

    accuracy                           0.99      9823
   macro avg       0.96      0.96      0.96      9823
weighted avg       0.99      0.99      0.99      9823



In [13]:
# =====================================
# Confusion Matrix
# =====================================

from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

print(cm)

[[ 990    2    0    2    0    0    0    0    0    0    5    1]
 [   0  391    0    0    0    0    0    0    0    0    0    0]
 [   0    0 1000    0    0    0    0    0    0    0    0    0]
 [   0    0    0  997    0    3    0    0    0    0    0    0]
 [   0    0    0    0 1000    0    0    0    0    0    0    0]
 [   0    0    0    0    0  992    6    0    0    0    1    1]
 [   2    0    0    0    0    1  993    0    0    0    3    1]
 [   0    0    0    0    0    0    0  999    0    0    1    0]
 [   0    0    0    0    0    0    0    0 1000    0    0    0]
 [   0    0    0    0    1    0    0    0    0  999    0    0]
 [   1    0    0    0    0    0    0    0    0    0  274   26]
 [   1    0    0    0    0    0    0    0    0    0   40   90]]


In [12]:
# =====================================
# Save Model
# =====================================

import joblib
import os

model_path = os.path.join(
    MODELS_PATH,
    "Improved_RandomForest_TrustXAI.pkl"
)

joblib.dump(rf_model, model_path)

print("Model saved successfully!")
print(model_path)

Model saved successfully!
D:\TrustXAI\models\Improved_RandomForest_TrustXAI.pkl


In [13]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average="weighted")
recall = recall_score(y_test, y_pred, average="weighted")
f1 = f1_score(y_test, y_pred, average="weighted")

print("=" * 50)
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print("=" * 50)

Accuracy : 0.9900
Precision: 0.9901
Recall   : 0.9900
F1 Score : 0.9900


In [14]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      0.99      0.99      1000
           1       0.99      1.00      1.00       391
           2       1.00      1.00      1.00      1000
           3       1.00      1.00      1.00      1000
           4       1.00      1.00      1.00      1000
           5       1.00      0.99      0.99      1000
           6       0.99      0.99      0.99      1000
           7       1.00      1.00      1.00      1000
          10       1.00      1.00      1.00      1000
          11       1.00      1.00      1.00      1000
          12       0.85      0.91      0.88       301
          14       0.76      0.69      0.72       131

    accuracy                           0.99      9823
   macro avg       0.96      0.96      0.96      9823
weighted avg       0.99      0.99      0.99      9823

